# <span style='color:blue'> LAB 6: </span>
# <span style='color:blue'> ATTENTION AND TRANSFORMER </span>

In [1]:
import torch
import time
import numpy as np
import seaborn as sns
import torchtext

## <span style='color:red'> IMDB Text Classification </span>

In [2]:
# Seaborn plot styling
sns.set(style = 'white', font_scale = 2)

## Prepare Data

In [3]:
import torchtext
from torch.utils.data import DataLoader
from collections import Counter

# Load dataset and initialize tokenizer
train_iter, test_iter = torchtext.datasets.IMDB(root='datasets', split=('train', 'test'))

label_counts = Counter()
for label, samples in train_iter:
    label_counts[label] += 1
print("Label distribution in train_iter:", label_counts)

label_counts = Counter()
for label, _ in test_iter:
    label_counts[label] += 1
print("Label distribution in test_iter:", label_counts)

TypeError: IMDB.__init__() missing 3 required positional arguments: 'path', 'text_field', and 'label_field'

In [ ]:
# Reload train_iter for use in DataLoader
#train_iter, test_iter = torchtext.datasets.IMDB(root='datasets', split=('train', 'test'))
tokenizer = torchtext.data.utils.get_tokenizer("basic_english")

def yield_tokens(data_iter):
    for _, text in data_iter:
        yield tokenizer(text)

# Create vocabulary with special tokens for padding and unknown words
vocab = torchtext.vocab.build_vocab_from_iterator(yield_tokens(train_iter), specials=["<unk>", "<pad>"])
vocab.set_default_index(vocab["<unk>"])

NameError: name 'train_iter' is not defined

In [ ]:
def text_pipeline(x): 
    return vocab(tokenizer(x))
    
# Example: Test the pipeline on a sample text
sample_text = "This movie was fantastic!"
print(text_pipeline(sample_text))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Function to calculate the length of each text in the dataset
def get_text_lengths(data_iter):
    lengths = []
    for _, text in tqdm(data_iter):
        tokenized_text = tokenizer(text)
        lengths.append(len(tokenized_text))
    return np.array(lengths)

# Load IMDB train dataset
train_iter, test_iter = torchtext.datasets.IMDB(root='datasets', split=('train', 'test'))

# Get text lengths from the entire dataset
train_text_lengths = get_text_lengths(train_iter)
test_text_lengths = get_text_lengths(test_iter)
# Plot the distribution of text lengths

# Define bin edges using the range of both datasets
min_bin = min(train_text_lengths.min(), test_text_lengths.min())
max_bin = max(train_text_lengths.max(), test_text_lengths.max())
bins = np.linspace(min_bin, max_bin, 30)  # Adjust number of bins as needed

plt.figure(figsize=(10, 6))
plt.subplot(2,1,1)
sns.histplot(train_text_lengths, kde=True, bins=50)
plt.xlabel('Train Text Length')
plt.ylabel('Frequency')
plt.subplot(2,1,2)
sns.histplot(test_text_lengths, kde=True, bins=50)
plt.xlabel('Test Text Length')
plt.ylabel('Frequency')
plt.suptitle('Distribution of Text Lengths in the IMDB Dataset')
plt.show()

In [ ]:
from torch.nn.utils.rnn import pad_sequence

# Define collate function for padding and batching
# setting a max_seq_len helps with estimating the max gpu memory usage
def collate_batch(batch, max_seq_len=1024):
    labels, texts = zip(*batch)
    # the labels start at 1 but predictions start at 0. To align them, we modify lables
    labels = torch.tensor(labels,dtype=torch.long)-1
    
    text_list = []
    for text in texts:
        # Truncate or pad to max_seq_len
        tokenized_text = text_pipeline(text)
        if len(tokenized_text) > max_seq_len:
            tokenized_text = tokenized_text[:max_seq_len]  # Truncate if longer than max_seq_len
        else:
            # Pad if shorter than max_seq_len
            tokenized_text = tokenized_text + [vocab["<pad>"]] * (max_seq_len - len(tokenized_text))
        
        text_list.append(torch.tensor(tokenized_text, dtype=torch.long))
    
    padded_texts = torch.stack(text_list)  # Stack the sequences into a tensor
    return padded_texts, labels

In [ ]:
from torch.utils.data import Dataset

class IMDBDataset(Dataset):
    def __init__(self, data_iter):
        self.data_iter = list(data_iter)  # Converting the iterator to a list for easier access

    def __len__(self):
        return len(self.data_iter)

    def __getitem__(self, idx):
        label, text = self.data_iter[idx]
        return label, text

train_iter, test_iter = torchtext.datasets.IMDB(root='datasets', split=('train', 'test'))
train_dataset = IMDBDataset(train_iter)
test_dataset = IMDBDataset(test_iter)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

# Train_loader and test_loader sanity check
# Initialize counter
label_counter = Counter()

# Iterate through batches in train_loader
for texts, labels in train_loader:
    label_counter.update(labels.tolist())
print("Label counts:", label_counter)

## Define Model

In [ ]:
class TransformerModel(torch.nn.Module):
    def __init__(self, vocab_size, embed_size, num_heads, num_encoder_layers, num_classes, dropout=0.1):
        
        super(TransformerModel, self).__init__()
        
        self.embedding = torch.nn.Embedding(vocab_size, embed_size)
        self.transformer = torch.nn.TransformerEncoder(
            torch.nn.TransformerEncoderLayer(embed_size, num_heads, embed_size * 2, dropout),
            num_encoder_layers
        )
        self.fc = torch.nn.Linear(embed_size, num_classes)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x):
        x = self.embedding(x)  # Embedding layer
        x = x.permute(1, 0, 2)  # Transformer expects (seq_len, batch_size, embedding_size)
        x = self.transformer(x)  # Apply transformer
        x = x.mean(dim=0)  # Pooling (take the mean of all tokens in the sequence)
        x = self.dropout(x)
        x = self.fc(x)  # Final classification layer
        return x

## Define Hyperparameters

In [ ]:
# Check for device compatibility, prioritizing CUDA, then MPS for MacBooks with Apple Silicon, and defaulting to CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Initialize the model, 
embed_size = 32
num_heads = 4
num_encoder_layers = 2
num_classes = 2  # Positive or negative sentiment
model = TransformerModel(len(vocab), embed_size, num_heads, num_encoder_layers, num_classes)

# Initialize loss function, and optimizer
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.to(device)

## Identify Tracked values

In [ ]:
num_epochs = 5
train_losses = np.zeros(num_epochs)
train_accuracies = np.zeros(num_epochs)

test_losses = np.zeros(num_epochs)
test_accuracies = np.zeros(num_epochs)

## Train Model

In [ ]:
def train_epoch(model, train_loader, loss_fn, optimizer):
    model.train()
    epoch_loss = 0
    epoch_accuracy = 0
    # total_batches = 0
    total_batches = len(train_loader)
    
    for texts, labels in tqdm(train_loader):
        texts, labels = texts.to(device), labels.to(device)
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(texts)
        
        # Compute loss and gradients
        loss = loss_fn(outputs, labels)
        loss.backward()
        
        # Update model parameters
        optimizer.step()
        
        # Calculate accuracy
        preds = torch.argmax(outputs, dim=1)
        correct = (preds == labels).sum().item()
        accuracy = correct / labels.size(0)
        
        epoch_loss += loss.item()
        epoch_accuracy += accuracy
    
    return epoch_loss / total_batches, epoch_accuracy / total_batches

def evaluate(model, test_loader, loss_fn):
    model.eval()
    epoch_loss = 0
    epoch_accuracy = 0
    total_batches = len(test_loader)
    
    with torch.no_grad():
        for texts, labels in tqdm(test_loader):
            texts, labels = texts.to(device), labels.to(device)
            # Forward pass
            outputs = model(texts)
            
            # Compute loss
            loss = loss_fn(outputs, labels)
            
            # Calculate accuracy
            preds = torch.argmax(outputs, dim=1)
            correct = (preds == labels).sum().item()
            accuracy = correct / labels.size(0)
            
            epoch_loss += loss.item()
            epoch_accuracy += accuracy
    
    return epoch_loss / total_batches, epoch_accuracy / total_batches

In [ ]:
for epoch in range(num_epochs):
    start_time = time.time()
    
    # Train for one epoch
    train_loss, train_accuracy = train_epoch(model, train_loader, loss_fn, optimizer)
    train_losses[epoch] = train_loss
    train_accuracies[epoch] = train_accuracy
    
    # Evaluate on the test set
    test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)
    test_losses[epoch] = test_loss
    test_accuracies[epoch] = test_accuracy
    end_time = time.time()
    
    print(f"Epoch [{epoch+1}/{num_epochs}] | Time: {end_time - start_time:.2f}s")
    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f}")
    print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_accuracy:.4f}")

## Visualize and Evaluate Model

In [ ]:
num_epochs = 5
epochs = list(range(0, num_epochs))

# Create 2x2 grid of subplots using plt.subplot
plt.figure(figsize=(10, 8))

# Training Loss
plt.subplot(2, 2, 1)  # (rows, columns, index)
plt.plot(epochs, train_losses, color='blue', label='Train Loss')
plt.title('Training Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Testing Loss
plt.subplot(2, 2, 2)
plt.plot(epochs, test_losses, color='orange', label='Test Loss')
plt.title('Testing Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# Training Accuracy
plt.subplot(2, 2, 3)
plt.plot(epochs, train_accuracies, color='green', label='Train Accuracy')
plt.title('Training Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Testing Accuracy
plt.subplot(2, 2, 4)
plt.plot(epochs, test_accuracies, color='red', label='Test Accuracy')
plt.title('Testing Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Adjust layout and display
plt.tight_layout()
plt.show()